In [ ]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

const bb = 120
const aa = 40
const N = 1_000_000
const I0 = 10
const n_iter_per_k = 20000
const KMAX_UPPER    = 30

include("functions.jl")

Random.seed!(2025)

data_file_for(tag::Symbol) =
    tag === :memoryless  ? "data/memoryless.csv" :
    tag === :sliding     ? "data/sliding_kmax14.csv" :
    tag === :powerlaw    ? "data/powerlaw_lambdaP.csv" :
    tag === :exponential ? "data/exponential_lambdaE.csv" :
    tag === :reciprocal  ? "data/reciprocal_lambdaR.csv" :
    error("Unknown data tag: $tag")

model_tag_sym = :sliding
data_tag_sym  = :sliding

data_path = data_file_for(data_tag_sym)
raw, hdr = DelimitedFiles.readdlm(data_path, ',', header=true)
raw = Matrix{Float64}(raw)
tau = size(raw, 2) ÷ 3

out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = String[]
if model_tag_sym === :memoryless
    header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    header_cont = ["beta", "alpha", "gamma", "lambda_R"]
elseif model_tag_sym === :sliding
    header_cont = ["beta", "alpha", "gamma"]
end

i = 10
c = 1
Istar_obs = Vector{Int}(round.(Int, raw[i, 1:tau]))

t_all = Dates.now()
for k in 1:KMAX_UPPER
    Random.seed!(2025 + i*100 + c*10_000 + k)
    initθ_chain = initθ_for_chain(model_tag_sym)

    t0 = Dates.now()
    try
        samples, loglik_aug_vecs =
            mcmc_one_chain_with_Rstar!(Istar_obs, N, I0;
                fit_mech      = model_tag_sym,
                n_iter        = n_iter_per_k,
                initθ         = initθ_chain,
                KMAX_UPPER    = KMAX_UPPER,
                k_max_fixed   = k)

        if size(samples, 1) == 0
            error("Empty samples returned for k=$k.")
        end

        samples_filename = "samples_sim_$(i)_chain_$(c)_k$(k).csv"
        write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

        loglik_filename = "loglik_sim_$(i)_chain_$(c)_k$(k).csv"
        write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))

        el = Dates.value(Dates.now() - t0) / 1000
        @info(@sprintf("Dataset %02d chain %d, k=%d: ok in %.2fs", i, c, k, el))
    catch err
        el = Dates.value(Dates.now() - t0) / 1000
        @warn(@sprintf("Dataset %02d chain %d, k=%d: ERROR - %s after %.2fs", i, c, k, err, el))
        continue
    end
end

el_all = Dates.value(Dates.now() - t_all) / 1000
@info(@sprintf("All k (1..%d) finished for dataset %02d. Total time %.2fs -> output dir: %s",
               KMAX_UPPER, i, el_all, out_dir))


[ Info: [sliding] iter 1000/20000 elapsed=3.2s, rate=0.349, medians=[0.617, 0.00921, 0.335], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 2000/20000 elapsed=8.3s, rate=0.298, medians=[0.630, 0.01216, 0.416], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 3000/20000 elapsed=11.5s, rate=0.312, medians=[0.682, 0.01386, 0.514], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 4000/20000 elapsed=14.8s, rate=0.345, medians=[0.749, 0.01521, 0.641], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 5000/20000 elapsed=18.0s, rate=0.383, medians=[0.818, 0.01644, 0.804], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 6000/20000 elapsed=21.3s, rate=0.416, medians=[0.879, 0.01762, 0.984], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 7000/20000 elapsed=24.5s, rate=0.443, medians=[0.928, 0.01871, 1.157], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 8000/20000 elapsed=27.7s, rate=0.463, medians=[0